# Import Modules

In [113]:
import numpy as np
import pandas as pd 

from collections import Counter
from sklearn.metrics import confusion_matrix

from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss                  
import matplotlib.pyplot as plt

import seaborn as sns
import tensorflow as tf                   

from tensorflow.keras.initializers import he_normal                                     
from sklearn.model_selection import train_test_split       

from tensorflow.keras import regularizers                                             
from joblib import dump, load               
                                            
from sklearn.preprocessing import StandardScaler                                        

# Choosing lead time

In [114]:
lead_time = 6

# Import and split data

In [115]:
test = pd.read_csv("/home/users/mendrika/NCAST/Data/Dakar/data-test-dakar.csv")
train_full = pd.read_csv("/home/users/mendrika/NCAST/Data/Dakar/data-train-dakar.csv")

In [116]:
val   = train_full[train_full["year"] == 2019].copy()
train = train_full[train_full["year"] != 2019].copy()

In [117]:
train

,year,month,day,hour,minute,lat1,lat2,lat3,lon1,lon2,...,mask1,mask2,mask3,Cb_dakar_t0,Cb_dakar_t1,Cb_dakar_t2,Cb_dakar_t3,Cb_dakar_t4,Cb_dakar_t5,Cb_dakar_t6
0,2016,6,1,0,0,9.117052,5.968581,7.792680,-13.006007,-10.854632,...,1,0,0,0,0,0,0,0,0,0
1,2016,6,1,0,15,9.117916,23.603060,20.197540,-13.120132,-12.231322,...,1,0,0,0,0,0,0,0,0,0
2,2016,6,1,0,30,9.204070,21.798041,20.956625,-13.466753,-11.361980,...,1,0,0,0,0,0,0,0,0,0
3,2016,6,1,0,45,9.176445,9.089847,8.058894,-13.494109,-13.090345,...,1,1,0,0,0,0,0,0,0,0
4,2016,6,1,1,0,9.176445,8.753265,6.555203,-13.494109,-12.677598,...,1,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112185,2004,9,30,18,15,10.802887,22.877389,6.204820,-14.093934,-12.951414,...,1,0,0,0,0,0,0,0,0,0
112186,2004,9,30,18,30,11.058403,19.986454,12.012427,-14.368961,-25.627664,...,1,0,0,0,0,0,0,0,0,0
112187,2004,9,30,18,45,11.031423,20.732701,8.757046,-14.483280,-25.877825,...,1,0,0,0,0,0,0,0,0,0
112188,2004,9,30,19,0,11.060403,20.853637,7.609722,-14.571967,-25.918951,...,1,0,0,0,0,0,0,0,0,0


# Selecting Header

In [118]:
def choose_header(n, location, lead_time):
    """
    Generate header names for n closest storms.

    Returns:
        tuple of two lists:
            - input headers: year, month, day, hour, minute, lat1..n, lon1..n, wp1..n, size1..n, d1..n, mask1..n
            - target header: Cb_{location}_t{lead_time}
    """
    input_headers = ['year', 'month', 'day', 'hour', 'minute']
    
    for prefix in ['lat', 'lon', 'wp', 'size', 'd', 'mask']:
        input_headers.extend([f'{prefix}{i}' for i in range(1, n + 1)])
    
    target_header = f'Cb_{location}_t{lead_time}'
    
    return input_headers, target_header

In [119]:
input_headers, target_header = choose_header(3, "dakar", lead_time)

In [120]:
X_train = train[input_headers].copy()
X_val   = val[input_headers].copy()
X_test  = test[input_headers].copy()

y_train = train[target_header]
y_val   = val[target_header]
y_test  = test[target_header]

In [121]:
X_train

,year,month,day,hour,minute,lat1,lat2,lat3,lon1,lon2,...,wp3,size1,size2,size3,d1,d2,d3,mask1,mask2,mask3
0,2016,6,1,0,0,9.117052,5.968581,7.792680,-13.006007,-10.854632,...,0.0,1701,0,0,789.374126,1211.769889,1194.218543,1,0,0
1,2016,6,1,0,15,9.117916,23.603060,20.197540,-13.120132,-12.231322,...,0.0,1440,0,0,781.730084,1130.503040,1072.143181,1,0,0
2,2016,6,1,0,30,9.204070,21.798041,20.956625,-13.466753,-11.361980,...,0.0,1152,0,0,751.657623,1017.336340,1151.994820,1,0,0
3,2016,6,1,0,45,9.176445,9.089847,8.058894,-13.494109,-13.090345,...,0.0,684,171,0,752.464500,786.188758,1219.004327,1,1,0
4,2016,6,1,1,0,9.176445,8.753265,6.555203,-13.494109,-12.677598,...,0.0,738,189,0,752.464500,843.434783,1058.296346,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112185,2004,9,30,18,15,10.802887,22.877389,6.204820,-14.093934,-12.951414,...,0.0,171,0,0,568.506415,1024.162671,1151.990230,1,0,0
112186,2004,9,30,18,30,11.058403,19.986454,12.012427,-14.368961,-25.627664,...,0.0,558,0,0,527.487215,1045.285527,1047.915334,1,0,0
112187,2004,9,30,18,45,11.031423,20.732701,8.757046,-14.483280,-25.877825,...,0.0,855,0,0,522.049849,1113.502017,1001.958732,1,0,0
112188,2004,9,30,19,0,11.060403,20.853637,7.609722,-14.571967,-25.918951,...,0.0,1098,0,0,513.584774,1124.830630,1259.941383,1,0,0


# Log transforming

In [122]:
cols_to_log = ['size1', 'size2', 'size3', 
               'wp1', 'wp2', 'wp3', 
               'd1', 'd2', 'd3']

for col in cols_to_log:
    if col in X_train.columns:
        X_train[col] = np.log1p(X_train[col])
        X_val[col]   = np.log1p(X_val[col])
        X_test[col]  = np.log1p(X_test[col])

In [123]:
X_train

,year,month,day,hour,minute,lat1,lat2,lat3,lon1,lon2,...,wp3,size1,size2,size3,d1,d2,d3,mask1,mask2,mask3
0,2016,6,1,0,0,9.117052,5.968581,7.792680,-13.006007,-10.854632,...,0.0,7.439559,0.000000,0.0,6.672506,7.100662,7.086084,1,0,0
1,2016,6,1,0,15,9.117916,23.603060,20.197540,-13.120132,-12.231322,...,0.0,7.273093,0.000000,0.0,6.662788,7.031302,6.978347,1,0,0
2,2016,6,1,0,30,9.204070,21.798041,20.956625,-13.466753,-11.361980,...,0.0,7.050123,0.000000,0.0,6.623610,6.925926,7.050118,1,0,0
3,2016,6,1,0,45,9.176445,9.089847,8.058894,-13.494109,-13.090345,...,0.0,6.529419,5.147494,0.0,6.624682,6.668468,7.106610,1,1,0
4,2016,6,1,1,0,9.176445,8.753265,6.555203,-13.494109,-12.677598,...,0.0,6.605298,5.247024,0.0,6.624682,6.738668,6.965360,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112185,2004,9,30,18,15,10.802887,22.877389,6.204820,-14.093934,-12.951414,...,0.0,5.147494,0.000000,0.0,6.344770,6.932607,7.050114,1,0,0
112186,2004,9,30,18,30,11.058403,19.986454,12.012427,-14.368961,-25.627664,...,0.0,6.326149,0.000000,0.0,6.270019,6.953002,6.955512,1,0,0
112187,2004,9,30,18,45,11.031423,20.732701,8.757046,-14.483280,-25.877825,...,0.0,6.752270,0.000000,0.0,6.259677,7.016163,6.910710,1,0,0
112188,2004,9,30,19,0,11.060403,20.853637,7.609722,-14.571967,-25.918951,...,0.0,7.002156,0.000000,0.0,6.243360,7.026276,7.139614,1,0,0


# Scaling

In [124]:
mask_cols = ['mask1', 'mask2', 'mask3']

# Columns grouped per core
core_groups = {
    1: ['lat1', 'lon1', 'wp1', 'size1', 'd1'],
    2: ['lat2', 'lon2', 'wp2', 'size2', 'd2'],
    3: ['lat3', 'lon3', 'wp3', 'size3', 'd3']
}

# Copy datasets
X_train_scaled = X_train.copy()
X_val_scaled   = X_val.copy()
X_test_scaled  = X_test.copy()

scalers = {}

for i, cols in core_groups.items():
    mask = X_train[f'mask{i}'] == 1
    
    scaler = StandardScaler()
    scaler.fit(X_train.loc[mask, cols])  # fit only on real cores
    
    scalers[i] = scaler
    
    # Apply to full dataset (including masked rows)
    X_train_scaled.loc[:, cols] = scaler.transform(X_train[cols])
    X_val_scaled.loc[:, cols]   = scaler.transform(X_val[cols])
    X_test_scaled.loc[:, cols]  = scaler.transform(X_test[cols])

# Model design

In [125]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(16, activation='relu'),

    tf.keras.layers.Dense(1, activation='sigmoid')
])

METRICS = [                      
    tf.keras.metrics.AUC(name='auc'),
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=METRICS
)

In [126]:
model.summary()

Model: "sequential_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_32 (Dense)            (None, 128)               3072      
                                                                 
 dropout_18 (Dropout)        (None, 128)               0         
                                                                 
 dense_33 (Dense)            (None, 64)                8256      
                                                                 
 dropout_19 (Dropout)        (None, 64)                0         
                                                                 
 dense_34 (Dense)            (None, 32)                2080      
                                                                 
 dropout_20 (Dropout)        (None, 32)                0         
                                                                 
 dense_35 (Dense)            (None, 16)               

In [127]:
EPOCHS = 200
BATCH_SIZE = 128

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc", 
    verbose=1,
    patience=10,
    mode='max',
    restore_best_weights=True)


# train the model
model_history = model.fit(
    X_train_scaled.to_numpy().astype("float32"),
    y_train.astype("float32").to_numpy(),
    epochs=EPOCHS,
    callbacks=[early_stopping],
    batch_size=BATCH_SIZE,
    validation_data=(
        X_val_scaled.to_numpy().astype("float32"),
        y_val.astype("float32").to_numpy()
    ),
)

Epoch 1/200


827/827 [==============================] - 7s 7ms/step - loss: 1.0962 - auc: 0.4994 - val_loss: 0.0767 - val_auc: 0.5000
Epoch 2/200
827/827 [==============================] - 6s 7ms/step - loss: 0.0824 - auc: 0.5065 - val_loss: 0.0768 - val_auc: 0.5000
Epoch 3/200
827/827 [==============================] - 3s 4ms/step - loss: 0.0690 - auc: 0.4941 - val_loss: 0.0809 - val_auc: 0.4935
Epoch 4/200
827/827 [==============================] - 4s 5ms/step - loss: 0.0652 - auc: 0.5136 - val_loss: 0.0792 - val_auc: 0.5000
Epoch 5/200
827/827 [==============================] - 5s 6ms/step - loss: 0.0630 - auc: 0.5130 - val_loss: 0.0764 - val_auc: 0.5084
Epoch 6/200
827/827 [==============================] - 5s 6ms/step - loss: 0.0626 - auc: 0.5104 - val_loss: 0.0760 - val_auc: 0.5155
Epoch 7/200
827/827 [==============================] - 6s 7ms/step - loss: 0.0618 - auc: 0.5108 - val_loss: 0.0757 - val_auc: 0.7324
Epoch 8/200
827/827 [==============================] - 5s 6ms/step - loss: 0.0611

# Model evaluation

In [128]:
# Predict probabilities
y_train_prob = model.predict(X_train_scaled)
y_val_prob   = model.predict(X_val_scaled)
y_test_prob  = model.predict(X_test_scaled)

# ROC AUC
train_auc = roc_auc_score(y_train, y_train_prob)
val_auc   = roc_auc_score(y_val, y_val_prob)
test_auc  = roc_auc_score(y_test, y_test_prob)

# Brier score
train_brier = brier_score_loss(y_train, y_train_prob)
val_brier   = brier_score_loss(y_val, y_val_prob)
test_brier  = brier_score_loss(y_test, y_test_prob)

print(f"Train ROC AUC : {train_auc:.4f}")
print(f"Val ROC AUC   : {val_auc:.4f}")
print(f"Test ROC AUC  : {test_auc:.4f}")

print()

print(f"Train Brier   : {train_brier:.4f}")
print(f"Val Brier     : {val_brier:.4f}")
print(f"Test Brier    : {test_brier:.4f}")

   1/3305 [..............................] - ETA: 3:24

938/938 [==============================] - 0s 438us/step
Train ROC AUC : 0.7859
Val ROC AUC   : 0.8581
Test ROC AUC  : 0.7503

Train Brier   : 0.0101
Val Brier     : 0.0141
Test Brier    : 0.0115
